# BÀI THỰC HÀNH CHƯƠNG 4 — NAMED ENTITY RECOGNITION & INFORMATION EXTRACTION

**Môn:** Xử lý ngôn ngữ tự nhiên  
**Chủ đề:** NER, BIO/BIOES, Gazetteer, Feature-based NER, CRF, đánh giá NER, Relation Extraction, Event/Template Filling

## Mục tiêu

Sau notebook này, sinh viên có thể:

1. Biểu diễn nhãn NER theo BIO.
2. Chuyển đổi giữa **token labels** và **entity spans**.
3. Kiểm tra một chuỗi BIO có hợp lệ hay không.
4. Xây dựng baseline NER bằng **gazetteer**.
5. Thiết kế các **feature bề mặt, ngôn ngữ và ngữ cảnh**.
6. Huấn luyện mô hình **Linear-chain CRF** cho NER.
7. Đánh giá NER bằng **token accuracy, exact span Precision/Recall/F1**.
8. Phân tích **strict matching** và **partial matching**.
9. Trích xuất **relation** và **event arguments** ở mức cơ bản.
10. Thực hiện **template filling** từ văn bản.

> Notebook được thiết kế theo hướng: **mẫu → TODO → đánh giá → phân tích lỗi**.

## Quy định bài làm

- Không thay đổi phần dữ liệu gốc trừ khi bài yêu cầu.
- Các ô có `TODO` là phần sinh viên phải hoàn thiện.
- Mỗi hàm cần có mô tả ngắn hoặc comment giải thích ý tưởng.
- Không chỉ báo cáo F1; phải có **error analysis**.
- Với các bài nâng cao, sinh viên có thể sử dụng thư viện ngoài nhưng phải ghi rõ.

### Thang điểm gợi ý

| Phần | Nội dung | Điểm |
|---|---|---:|
| 1 | BIO + span conversion + validator | 1.5 |
| 2 | Gazetteer NER | 1.5 |
| 3 | Feature engineering + CRF | 2.5 |
| 4 | Đánh giá + phân tích lỗi | 1.5 |
| 5 | Relation Extraction | 1.0 |
| 6 | Event/Template Filling | 1.5 |
| 7 | Bài nâng cao | +1.0 bonus |

# 0. Chuẩn bị môi trường

In [1]:
from collections import defaultdict, Counter
import re
import math
import json
from pprint import pprint

print("Environment ready.")

Environment ready.


Notebook sử dụng `sklearn-crfsuite` cho phần CRF. Nếu chưa cài, chạy ô sau.

In [ ]:
# %pip install -q sklearn-crfsuite seqeval

In [2]:

import sklearn_crfsuite
import seqeval

# 1. Dữ liệu NER mẫu

Ta sử dụng một tập dữ liệu nhỏ để minh họa. Mỗi câu được biểu diễn thành `(tokens, BIO_labels)`.

Các loại thực thể:
- `PER`: Person
- `ORG`: Organization
- `LOC`: Location
- `DATE`: Date/Time
- `MONEY`: Money
- `POSITION`: Position

In [3]:
train_data = [
    (["Công_ty", "Sao_Bắc", "bổ_nhiệm", "bà", "Linh", "làm", "giám_đốc", "."],
     ["B-ORG", "I-ORG", "O", "O", "B-PER", "O", "B-POSITION", "O"]),
    (["FPT", "đặt", "trụ_sở", "tại", "Hà_Nội", "."],
     ["B-ORG", "O", "O", "O", "B-LOC", "O"]),
    (["Nguyễn_Văn_An", "làm_việc", "tại", "Viettel", "."],
     ["B-PER", "O", "O", "B-ORG", "O"]),
    (["VinFast", "mở", "nhà_máy", "mới", "ở", "Hải_Phòng", "."],
     ["B-ORG", "O", "O", "O", "O", "B-LOC", "O"]),
    (["Đại_học", "Quốc_gia", "Hà_Nội", "tổ_chức", "hội_thảo", "."],
     ["B-ORG", "I-ORG", "I-ORG", "O", "O", "O"]),
    (["Bà", "Mai", "đến", "Đà_Nẵng", "vào", "12/09/2026", "."],
     ["O", "B-PER", "O", "B-LOC", "O", "B-DATE", "O"]),
    (["Công_ty", "ABC", "tuyển", "kỹ_sư", "tại", "TP.HCM", "."],
     ["B-ORG", "I-ORG", "O", "O", "O", "B-LOC", "O"]),
    (["Apple", "mở", "văn_phòng", "tại", "Singapore", "."],
     ["B-ORG", "O", "O", "O", "B-LOC", "O"]),
]

test_data = [
    (["Công_ty", "Minh_Long", "bổ_nhiệm", "ông", "Nam", "làm", "phó_giám_đốc", "."],
     ["B-ORG", "I-ORG", "O", "O", "B-PER", "O", "B-POSITION", "O"]),
    (["Viettel", "mở", "chi_nhánh", "tại", "Đà_Nẵng", "."],
     ["B-ORG", "O", "O", "O", "B-LOC", "O"]),
    (["Lan", "gia_nhập", "FPT", "vào", "01/10/2026", "."],
     ["B-PER", "O", "B-ORG", "O", "B-DATE", "O"])
]

print("Train sentences:", len(train_data))
print("Test sentences :", len(test_data))

Train sentences: 8
Test sentences : 3


# 2. Bài 1 — BIO, span annotation và kiểm tra nhãn

## 2.1. Hàm mẫu: tách prefix và entity type

In [4]:
def split_bio_label(label):
    if label == "O":
        return "O", None
    prefix, entity_type = label.split("-", 1)
    return prefix, entity_type

for label in ["B-ORG", "I-PER", "O"]:
    print(label, "->", split_bio_label(label))

B-ORG -> ('B', 'ORG')
I-PER -> ('I', 'PER')
O -> ('O', None)


## 2.2. TODO 1 — Viết BIO validator

Một chuỗi BIO hợp lệ cần thỏa:
- `I-X` không được đứng đầu chuỗi.
- `I-X` chỉ được đi sau `B-X` hoặc `I-X`.
- `B-X` có thể bắt đầu một thực thể mới.
- `O` có thể xuất hiện ở bất kỳ vị trí nào.

Ví dụ:
```text
B-ORG I-ORG O B-PER   -> hợp lệ
O I-ORG O             -> không hợp lệ
B-ORG I-PER O         -> không hợp lệ
```

In [5]:
def validate_bio(labels):
    """Trả về (True, None) nếu hợp lệ, ngược lại (False, message)."""
    previous_prefix = None
    previous_entity_type = None

    for i, label in enumerate(labels):
        prefix, entity_type = split_bio_label(label)
        if prefix == "O":
            previous_prefix = "O"
            previous_entity_type = None
            continue
        if prefix == "B":
            previous_prefix = "B"
            previous_entity_type = entity_type
            continue
        if prefix == "I":
            if previous_prefix not in ("B", "I"):
                return False, f"I-{entity_type} tại vị trí {i} không thể đứng sau {previous_prefix}"
            if previous_entity_type != entity_type:
                return False, (
                    f"I-{entity_type} tại vị trí {i} "
                    f"không thể đứng sau {previous_prefix}-{previous_entity_type}"
                )

            previous_prefix = "I"
            previous_entity_type = entity_type
            continue
        return False, f"Label không hợp lệ: {label}"
    return True, None

### Kiểm thử bắt buộc

In [8]:
# Bỏ comment sau khi hoàn thiện validate_bio
assert validate_bio(["B-ORG", "I-ORG", "O", "B-PER"])[0] is True
assert validate_bio(["O", "I-ORG", "O"])[0] is False
assert validate_bio(["B-ORG", "I-PER", "O"])[0] is False
assert validate_bio(["B-PER", "I-PER", "I-PER"])[0] is True

## 2.3. TODO 2 — Chuyển BIO labels → entity spans

Biểu diễn một span:
```python
{"start": 0, "end": 2, "type": "ORG", "text": "Công_ty Sao_Bắc"}
```
`end` dùng quy ước exclusive.

In [9]:
def bio_to_spans(tokens, labels):
    spans = []
    start = None
    entity_type = None

    for i, label in enumerate(labels):
        prefix, current_type = split_bio_label(label)
        if prefix == "B":
            if start is not None:
                spans.append({
                    "start": start,
                    "end": i,
                    "type": entity_type,
                    "text": " ".join(tokens[start:i])
                })

            start = i
            entity_type = current_type
        elif prefix == "I":
            continue
        elif prefix == "O":
            if start is not None:
                spans.append({
                    "start": start,
                    "end": i,
                    "type": entity_type,
                    "text": " ".join(tokens[start:i])
                })

                start = None
                entity_type = None
    if start is not None:
        spans.append({
            "start": start,
            "end": len(tokens),
            "type": entity_type,
            "text": " ".join(tokens[start:])
        })
    return spans

In [10]:
tokens = ["Công_ty", "Sao_Bắc", "đang", "tuyển", "dụng", "Nguyễn", "An"]
labels = ["B-ORG", "I-ORG", "O", "O", "O", "B-PER", "I-PER"]
print(bio_to_spans(tokens, labels))

[{'start': 0, 'end': 2, 'type': 'ORG', 'text': 'Công_ty Sao_Bắc'}, {'start': 5, 'end': 7, 'type': 'PER', 'text': 'Nguyễn An'}]


## 2.4. TODO 3 — Chuyển entity spans → BIO labels

In [11]:
def spans_to_bio(tokens, spans):
    labels = ["O"] * len(tokens)
    for span in spans:
        start = span["start"]
        end = span["end"]
        entity_type = span["type"]

        labels[start] = f"B-{entity_type}"
        for i in range(start + 1, end):
            labels[i] = f"I-{entity_type}"
    return labels

In [13]:
tokens = ["Công_ty", "Sao_Bắc", "đang", "tuyển", "Nguyễn", "An"]
spans = [
    {"start": 0, "end": 2, "type": "ORG", "text": "Công_ty Sao_Bắc"},
    {"start": 4, "end": 6, "type": "PER", "text": "Nguyễn An"}
]
print(spans_to_bio(tokens, spans))

['B-ORG', 'I-ORG', 'O', 'O', 'B-PER', 'I-PER']


## 2.5. Round-trip test

In [14]:
tokens, labels = train_data[0]
spans = bio_to_spans(tokens, labels)
labels_reconstructed = spans_to_bio(tokens, spans)
print(spans)
assert labels == labels_reconstructed

[{'start': 0, 'end': 2, 'type': 'ORG', 'text': 'Công_ty Sao_Bắc'}, {'start': 4, 'end': 5, 'type': 'PER', 'text': 'Linh'}, {'start': 6, 'end': 7, 'type': 'POSITION', 'text': 'giám_đốc'}]


# 3. Bài 2 — Gazetteer-based NER

Pipeline:
```text
Raw text → Normalize → Match gazetteer → Resolve overlap → Entity candidates
```

In [ ]:
gazetteer = {
    "ORG": {"fpt", "viettel", "vinfast", "apple", "công ty sao bắc", "công ty abc", "đại học quốc gia hà nội", "công ty minh long"},
    "LOC": {"hà nội", "hải phòng", "đà nẵng", "tp.hcm", "singapore"},
    "PER": {"linh", "mai", "nam", "lan", "nguyễn văn an"}
}

## 3.1. Normalize mẫu

In [ ]:
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

print(normalize_text("  Công ty   Sao Bắc  "))

## 3.2. TODO 4 — Gazetteer matching

Tìm tất cả entity candidates và trả về character offsets, ví dụ:
```python
[{"start": 0, "end": 16, "text": "Công ty Sao Bắc", "type": "ORG"}]
```

In [ ]:
def gazetteer_match(text, gazetteer):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 3.3. TODO 5 — Resolve overlap

Baseline: **Longest match wins**.

In [ ]:
def spans_overlap(a, b):
    return not (a["end"] <= b["start"] or b["end"] <= a["start"])


def resolve_overlap_longest(candidates):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 3.4. Thử nghiệm xuyên suốt

Dùng câu:
```text
Đại học Quốc gia Hà Nội mở trung tâm mới tại Hà Nội.
```

Giải thích: candidate nào overlap, candidate nào bị loại, và kết quả cuối có hợp lý không.

In [ ]:
sample_text = "Đại học Quốc gia Hà Nội mở trung tâm mới tại Hà Nội."
# candidates = gazetteer_match(sample_text, gazetteer)
# pprint(candidates)
# pprint(resolve_overlap_longest(candidates))

## 3.5. Câu hỏi phân tích

1. Vì sao gazetteer có precision cao nhưng recall có thể thấp?
2. Vì sao normalize quá mạnh có thể tạo false positive?
3. Khi nào `Longest match wins` có thể sai?
4. Nếu bài toán hỗ trợ **nested NER**, ta có nên luôn loại span nhỏ hơn không?

**TRẢ LỜI CỦA SINH VIÊN:**
- Câu 1:
- Câu 2:
- Câu 3:
- Câu 4:

# 4. Bài 3 — Feature-based NER và Linear-chain CRF

Các nhóm feature:
- bề mặt: chữ hoa/thường, chữ số, độ dài, shape;
- ngữ cảnh: token trước/sau, cửa sổ ±2;
- gazetteer;
- đặc trưng tiếng Việt/dữ liệu.

## 4.1. Feature extraction mẫu

In [ ]:
def token_features(sent, i):
    token = sent[i]
    low = token.lower()
    feats = {
        "bias": 1.0,
        "token.lower": low,
        "is_upper": token.isupper(),
        "is_title": token.istitle(),
        "has_digit": any(ch.isdigit() for ch in token),
        "length": len(token),
    }
    if i > 0:
        feats["prev.lower"] = sent[i - 1].lower()
    else:
        feats["BOS"] = True
    if i < len(sent) - 1:
        feats["next.lower"] = sent[i + 1].lower()
    else:
        feats["EOS"] = True
    return feats

print(token_features(train_data[0][0], 4))

## 4.2. TODO 6 — Mở rộng feature

Bổ sung ít nhất **5 feature** mới, bắt buộc có:
- một feature từ gazetteer;
- một feature về shape;
- một feature ngữ cảnh ±2;
- một feature đặc thù tiếng Việt/dữ liệu;
- một feature tự đề xuất.

In [ ]:
def token_features_improved(sent, i, gazetteer):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 4.3. Chuẩn bị dữ liệu CRF

In [ ]:
def sent2features(tokens, feature_fn):
    return [feature_fn(tokens, i) for i in range(len(tokens))]

X_train_basic = [sent2features(tokens, token_features) for tokens, labels in train_data]
y_train = [labels for tokens, labels in train_data]
X_test_basic = [sent2features(tokens, token_features) for tokens, labels in test_data]
y_test = [labels for tokens, labels in test_data]

## 4.4. Huấn luyện CRF baseline

CRF mô hình hóa trực tiếp:
\[
P(Y|X)
\]
và tìm:
\[
\hat{Y}=\arg\max_Y P(Y|X)
\]

In [ ]:
try:
    import sklearn_crfsuite
    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=0.1, c2=0.1,
        max_iterations=100,
        all_possible_transitions=True
    )
    crf.fit(X_train_basic, y_train)
    y_pred_basic = crf.predict(X_test_basic)
    print("CRF baseline trained.")
    for (tokens, gold), pred in zip(test_data, y_pred_basic):
        print("\nTOKENS:", tokens)
        print("GOLD  :", gold)
        print("PRED  :", pred)
except ImportError:
    print("Chưa có sklearn-crfsuite. Hãy chạy cell cài đặt ở phần 0.")

## 4.5. TODO 7 — CRF với feature cải tiến

1. Dùng `token_features_improved`.
2. Huấn luyện lại CRF.
3. So sánh với baseline.
4. Nêu ít nhất 3 feature hữu ích nhất theo quan sát của nhóm.

In [ ]:
# TODO
# X_train_improved = ...
# X_test_improved = ...
# crf_improved = ...
# crf_improved.fit(...)
# y_pred_improved = ...

# 5. Bài 4 — Đánh giá NER

### Token-level accuracy
\[
Accuracy = \frac{\text{số token dự đoán đúng nhãn}}{\text{tổng số token}}
\]

### Exact span matching
Một entity chỉ đúng khi `start`, `end`, `type` đều đúng.

### Strict vs Partial
- **Strict:** span + type khớp hoàn toàn.
- **Partial:** cho phép chồng lấp theo một quy ước định trước.

## 5.1. TODO 8 — Token accuracy

In [ ]:
def token_accuracy(y_true, y_pred):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 5.2. TODO 9 — Entity-level exact Precision/Recall/F1

\[
Precision = \frac{TP}{TP+FP},\quad
Recall = \frac{TP}{TP+FN},\quad
F1 = \frac{2PR}{P+R}
\]

In [ ]:
def entity_prf(tokens_list, y_true, y_pred):
    """Trả về TP, FP, FN, precision, recall, f1 theo exact span matching."""
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 5.3. TODO 10 — Partial matching

Gợi ý dùng IoU:
\[
IoU = \frac{|Gold \cap Pred|}{|Gold \cup Pred|}
\]
Match nếu cùng type và `IoU >= 0.5`.

In [ ]:
def span_iou(a, b):
    inter = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))
    union = max(a["end"], b["end"]) - min(a["start"], b["start"])
    return inter / union if union > 0 else 0.0


def partial_match_prf(tokens_list, y_true, y_pred, iou_threshold=0.5):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 5.4. Error analysis

Chọn ít nhất **5 lỗi** và phân loại: boundary, sai type, bỏ sót, false positive, unseen word, thiếu ngữ cảnh, tokenization, gazetteer.

| Câu | Gold | Prediction | Loại lỗi | Nguyên nhân | Cách cải tiến |
|---|---|---|---|---|---|
| ... | ... | ... | ... | ... | ... |

# 6. Bài 5 — Relation Extraction

Một relation thường được biểu diễn:
\[
(Entity_1, RelationType, Entity_2)
\]

Ví dụ:
```text
(Nguyễn Văn An, WORK_FOR, FPT)
```

In [ ]:
relation_examples = [
    "Nguyễn Văn An làm việc tại FPT.",
    "Lan gia nhập Viettel.",
    "VinFast đặt nhà máy tại Hải Phòng.",
    "Apple mở văn phòng tại Singapore."
]

## 6.1. TODO 11 — Trích xuất relation bằng rule

Xây dựng ít nhất 2 loại relation:
- `WORK_FOR(Person, Organization)`
- `LOCATED_IN(Organization, Location)`

In [ ]:
def extract_relations(text, gazetteer):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 6.2. Đánh giá relation

Tự tạo 5 câu gold, chạy hệ thống và tính Precision/Recall/F1.
Một relation chỉ đúng nếu entity 1, entity 2 và relation type đều đúng; với quan hệ có hướng, thứ tự entity cũng quan trọng.

# 7. Bài 6 — Event Extraction và Template Filling

Với event, đơn vị dự đoán thường gồm:
```text
Event Type + Trigger + Arguments và roles
```

Ví dụ:
```text
Công ty Sao Bắc bổ nhiệm bà Linh làm giám đốc vào ngày 12/09/2026.
```

Template:
```text
Event type: APPOINTMENT
Trigger: bổ nhiệm
Organization = Công ty Sao Bắc
Person = Linh
Position = giám đốc
Date = 12/09/2026
```

In [ ]:
event_texts = [
    "Công ty Sao Bắc bổ nhiệm bà Linh làm giám đốc vào ngày 12/09/2026.",
    "Công ty Minh Long bổ nhiệm ông Nam làm phó giám đốc vào ngày 01/10/2026."
]

## 7.1. TODO 12 — Event trigger detection

In [ ]:
def detect_event_trigger(text):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

Mở rộng ít nhất một event type khác, ví dụ `ACQUISITION`, `RECRUITMENT` hoặc `OPENING`.

## 7.2. TODO 13 — Slot filling

In [ ]:
def fill_appointment_template(text, gazetteer):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

## 7.3. Template Filling

- **Slot filling:** điền từng trường riêng lẻ.
- **Template filling:** kết hợp nhiều slot thành một record cấu trúc.

Chạy trên toàn bộ `event_texts` và xuất JSON.

In [ ]:
# TODO
# results = [fill_appointment_template(x, gazetteer) for x in event_texts]
# print(json.dumps(results, ensure_ascii=False, indent=2))

# 8. Bài nâng cao — Subword alignment cho Transformer NER

Ví dụ minh họa:
```text
VinFast -> Vin, ##Fast
```

Nếu word-level label là `B-ORG`:

**First-subword-only**
```text
Vin -> B-ORG
##Fast -> -100
```

**Label-all-subwords**
```text
Vin -> B-ORG
##Fast -> I-ORG
```

Special tokens như `[CLS]`, `[SEP]`, `[PAD]` thường gán `-100` để bỏ qua khi tính loss.

## 8.1. TODO 14 — Align labels với `word_ids`

Với:
```python
word_ids = [None, 0, 0, 1, 2, None]
labels = ["B-ORG", "O", "B-LOC"]
```

Yêu cầu:
- `None -> -100`
- subword đầu lấy label gốc
- subword tiếp theo: `-100` hoặc đổi `B-X -> I-X` tùy chiến lược.

In [ ]:
def align_labels_with_word_ids(word_ids, word_labels, first_subword_only=True):
    # TODO: Sinh viên hoàn thiện
    raise NotImplementedError

# 9. Bài bonus — So sánh HMM, CRF, BiLSTM-CRF, Transformer

| Mô hình | Input representation | Context | Quan hệ giữa nhãn | Feature thủ công | Ưu điểm | Hạn chế |
|---|---|---|---|---|---|---|
| HMM | | | | | | |
| CRF | | | | | | |
| BiLSTM-CRF | | | | | | |
| Transformer | | | | | | |

Trả lời:
1. Vì sao HMM là mô hình sinh còn CRF là mô hình phân biệt?
2. Vì sao BiLSTM giúp giảm nhu cầu feature engineering?
3. Vì sao CRF vẫn có thể hữu ích phía trên BiLSTM?
4. Transformer token classification có nhất thiết phải dùng CRF không?

# 10. Báo cáo kết quả cuối notebook

### 10.1. Kết quả
- Token accuracy:
- Exact entity Precision:
- Exact entity Recall:
- Exact entity F1:
- Partial entity F1:
- Relation F1:

### 10.2. Ba lỗi phổ biến nhất
1.
2.
3.

### 10.3. Ba cải tiến quan trọng nhất
1.
2.
3.

### 10.4. Nhận xét
Trong khoảng **150–250 từ**, nhận xét sự khác nhau giữa Gazetteer-based NER, Feature-based CRF và Neural/Transformer NER; giải thích khi nào nên dùng hệ thống hybrid.

# 11. Checklist trước khi nộp

- [ ] BIO validator chạy đúng.
- [ ] BIO ↔ spans chạy round-trip.
- [ ] Gazetteer matcher hoạt động.
- [ ] Resolve overlap hoạt động.
- [ ] Có ít nhất 5 feature mới cho CRF.
- [ ] Có baseline CRF và improved CRF.
- [ ] Có token accuracy.
- [ ] Có exact entity P/R/F1.
- [ ] Có partial matching.
- [ ] Có error analysis ít nhất 5 trường hợp.
- [ ] Có ít nhất 2 loại relation.
- [ ] Có event trigger + slot/template filling.
- [ ] Có phần nhận xét cuối.